# Лабораторная работа №2 Задача классификации

## Задание

Используйте классификатор SVM для данных находящихся пo
[ссылке](https://drive.google.com/file/d/1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec/view?usp=share_link) (данный датасет применялся в лабораторной работе №1). Здесь - выходящей меткой (учителем) является столбец / атрибут - `outcome`. Предоставьте результаты, используя следующие ядерные функции:
- Линейное ядро
- Полиномиальное ядро
- Гауссово ядро
- Сигмоидное ядро

Укажите `score` моделей. Сопоставьте полученные результаты и сделайте вывод.

## Загрузка данных

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

In [2]:
LOCAL_PATH = "data/diabetes.csv"
GITHUB_RAW_URL = "https://raw.githubusercontent.com/lufikow/intro-ai/refs/heads/main/data/diabetes.csv"

if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
else:
    df = pd.read_csv(GITHUB_RAW_URL)

In [3]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
Glucose,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
BloodPressure,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
SkinThickness,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
Insulin,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
BMI,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
DiabetesPedigreeFunction,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
Age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00
Outcome,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.00


In [6]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='str')

## Классификатор

In [7]:
df['Outcome'].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [8]:
def make_model(kernel):
    return make_pipeline(SimpleImputer(strategy='median'),
                         StandardScaler(),
                         SVC(kernel=kernel, random_state=42))

C_grid = [0.1, 1, 10, 100]
gamma_grid = ['scale', 0.001, 0.01, 0.1]
setups = {
    'Linear': (make_model('linear'), {'svc__C': C_grid}),
    'Polynomial': (make_model('poly'), {'svc__C': C_grid, 'svc__degree': [2, 3], 'svc__gamma': gamma_grid}),
    'Gaussian': (make_model('rbf'), {'svc__C': C_grid, 'svc__gamma': gamma_grid}),
    'Sigmoid': (make_model('sigmoid'), {'svc__C': C_grid, 'svc__gamma': gamma_grid})
}


In [9]:
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[zero_cols] = df[zero_cols].replace(0, np.nan)

X = df.drop(columns='Outcome')
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rows, reports = [], {}

for name, (pipe, grid) in setups.items():
    gs = GridSearchCV(pipe, grid, cv=5, scoring='accuracy')
    gs.fit(X_train, y_train)
    best = gs.best_estimator_
    pred = best.predict(X_test)
    proba = best.decision_function(X_test)
    params = {k.replace('svc__', ""): v for k, v in gs.best_params_.items()}
    rows.append({
        'kernel': name,
        'best params': params,
        'CV accuracy': gs.best_score_,
        'test score': accuracy_score(y_test, pred),
        'f1': f1_score(y_test, pred),
        'roc auc': roc_auc_score(y_test, proba)
    })
    reports[name] = classification_report(y_test, pred)

print(pd.DataFrame(rows).round(3))
for name, rep in reports.items():
    print(f'\n{name} \n {rep}')

       kernel                          best params  CV accuracy  test score  \
0      Linear                           {'C': 0.1}        0.775       0.708   
1  Polynomial  {'C': 1, 'degree': 3, 'gamma': 0.1}        0.728       0.708   
2    Gaussian           {'C': 100, 'gamma': 0.001}        0.777       0.695   
3     Sigmoid             {'C': 10, 'gamma': 0.01}        0.775       0.701   

      f1  roc auc  
0  0.536    0.813  
1  0.458    0.811  
2  0.525    0.817  
3  0.531    0.811  

Linear 
               precision    recall  f1-score   support

           0       0.75      0.83      0.79       100
           1       0.60      0.48      0.54        54

    accuracy                           0.71       154
   macro avg       0.68      0.66      0.66       154
weighted avg       0.70      0.71      0.70       154


Polynomial 
               precision    recall  f1-score   support

           0       0.72      0.90      0.80       100
           1       0.66      0.35      0.46 

## Вывод

* По `score` (accuracy) все четыре ядра дают близкий друг к другу результат (разборс между 0.695 и 0.708). Разница находится в пределах случайного разброса.
* По ROC AUC все модели также примерно одинаковы.
* Полиномиальное ядро слабо находит диабетиков. Несмотря на одинаковый `recall` с LinearRegression, названная модель лучше предсказывает диабетиков, что положительно сказывается на практической значимости: пропуск больного хуже ложной тревоги.
* Наиболее предпочительной моделью для датасета является Linear ввиду сбалансированных высоких метрик.